# MindSight — Multimodal Emotion & Mental Health Analysis
**Face + Text + Behaviour + Audio → Emotion + Mental Health Insights**

In [ ]:
# CELL 0 — INSTALL
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q",
    "transformers","torch","torchvision","torchaudio",
    "tensorflow-hub","librosa","soundfile",
    "scikit-learn","Pillow","groq","accelerate",
    "matplotlib","seaborn","pandas","numpy"], check=True)
print("Done")


In [ ]:
# CELL 1 — CONFIG
import os, json, pickle, warnings, re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import tensorflow as tf
from tensorflow import keras
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive", force_remount=False)

MODELS_DIR = "/content/drive/MyDrive/MindSight_Models"
FACE_V2    = "/content/drive/MyDrive/Emotion_Project/face_model/exports/face_emotion_model.keras"
FACE_V1    = MODELS_DIR + "/facial_model.keras"
BERT_DIR   = MODELS_DIR + "/bert_text_model"
BEH_PKL    = MODELS_DIR + "/behavioral_model.pkl"
BEH_COLS   = MODELS_DIR + "/behavioral_columns.json"
BEH_LABELS = MODELS_DIR + "/behavioral_labels.json"

EMOTIONS = ["angry","disgust","fear","happy","neutral","sad","surprise"]
EMOTION_TO_STATE = {
    "angry":"negative","disgust":"negative","fear":"negative","sad":"negative",
    "neutral":"neutral","happy":"positive","surprise":"positive"}
EMOTION_EMOJI = {"angry":"😠","disgust":"🤢","fear":"😨",
                 "happy":"😊","neutral":"😐","sad":"😢","surprise":"😲"}

# Weights including audio
WEIGHTS = {"face":0.40,"text":0.20,"behaviour":0.10,"audio":0.30}

GROQ_API_KEY = "YOUR_GROQ_API_KEY"  # get free key at console.groq.com
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# CELL 2 — LOAD MODELS
print("Loading models...")

# Face
if os.path.exists(FACE_V2):
    face_model = keras.models.load_model(FACE_V2, compile=False)
    FACE_SIZE  = 96; FACE_DUAL = len(face_model.outputs) > 1
else:
    face_model = keras.models.load_model(FACE_V1, compile=False)
    FACE_SIZE  = 48; FACE_DUAL = False
print(f"Face model: {face_model.name}")

# BERT
bert_tok   = BertTokenizer.from_pretrained(BERT_DIR)
bert_model = BertForSequenceClassification.from_pretrained(BERT_DIR,num_labels=7)
bert_model.eval(); bert_model.to(DEVICE)
print("BERT loaded")

# Behaviour
with open(BEH_PKL,"rb") as f: beh_model = pickle.load(f)
with open(BEH_COLS) as f: beh_cols = json.load(f)
with open(BEH_LABELS) as f: beh_lbls = json.load(f)
print("Behaviour model loaded")

# Audio — WavLM (load if available)
AUDIO_MODEL_PATH = MODELS_DIR + "/best_wavlm_ser_v2.pt"
audio_model = None
if os.path.exists(AUDIO_MODEL_PATH):
    try:
        import torch
        audio_model = torch.load(AUDIO_MODEL_PATH,
                                 map_location=torch.device("cpu"))
        audio_model.eval()
        print("WavLM audio model loaded")
    except Exception as e:
        print(f"Audio model load error: {e}")
        print("Audio modality will be skipped")
else:
    print("Audio model not found — audio modality skipped")
    print(f"Expected at: {AUDIO_MODEL_PATH}")

print("\nAll models loaded!")


In [ ]:
# CELL 3 — PREDICTION FUNCTIONS

def predict_face(image_path):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((FACE_SIZE,FACE_SIZE),Image.LANCZOS)
    arr = (np.array(img,dtype=np.float32)/127.5)-1.0
    p   = face_model.predict(arr[None,...],verbose=0)
    proba = p[0][0] if FACE_DUAL else p[0]
    proba = np.array(proba[:7],dtype=np.float32)
    return proba/(proba.sum()+1e-8)

def predict_text(text):
    enc = bert_tok(text,max_length=128,padding="max_length",
                   truncation=True,return_tensors="pt")
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    with torch.no_grad():
        logits = bert_model(**enc).logits
    return torch.softmax(logits,dim=-1).cpu().numpy()[0].astype(np.float32)

BEH_MAP = {"high"  :[0.25,0.15,0.20,0.05,0.10,0.20,0.05],
            "medium":[0.10,0.08,0.10,0.15,0.30,0.15,0.12],
            "low"   :[0.05,0.03,0.05,0.30,0.30,0.05,0.22]}

def predict_behaviour(responses):
    row  = [responses.get(c,0) for c in beh_cols]
    X    = np.array(row,dtype=np.float32).reshape(1,-1)
    pred = str(beh_model.predict(X)[0]).lower()
    if hasattr(beh_model,"predict_proba"):
        sp  = beh_model.predict_proba(X)[0]
        lo  = [l.lower() for l in beh_lbls]
        emo = sum(sp[i]*np.array(BEH_MAP.get(lo[i],BEH_MAP["medium"]))
                  for i in range(len(lo)))
    else:
        emo = np.array(BEH_MAP.get(pred,BEH_MAP["medium"]),dtype=np.float32)
    return (emo/(emo.sum()+1e-8)).astype(np.float32), pred

def predict_audio(audio_path):
    if audio_model is None:
        return None
    try:
        import librosa
        waveform,_ = librosa.load(audio_path,sr=16000,mono=True)
        waveform   = torch.tensor(waveform).unsqueeze(0)
        with torch.no_grad():
            logits = audio_model(waveform).logits
        return torch.softmax(logits,dim=-1).cpu().numpy()[0].astype(np.float32)
    except Exception as e:
        print(f"Audio prediction error: {e}")
        return None

def fuse(face_p=None,text_p=None,beh_p=None,audio_p=None):
    active = {k:v for k,v in
              {"face":face_p,"text":text_p,
               "behaviour":beh_p,"audio":audio_p}.items()
              if v is not None}
    total_w = sum(WEIGHTS[k] for k in active)
    fused   = sum(WEIGHTS[k]/total_w*p for k,p in active.items())
    return fused.astype(np.float32), {k:round(WEIGHTS[k]/total_w,3)
                                       for k in active}

print("Prediction functions ready")


In [ ]:
# CELL 4 — AI REPORT
def generate_report(emotion,state,conf,fused,
                    face_p=None,text_p=None,beh_p=None,
                    audio_p=None,stress=None,text_in=None,used_w=None):
    lines = []
    if face_p  is not None:
        lines.append(f"- Face    : {EMOTIONS[np.argmax(face_p)].upper()} ({face_p.max()*100:.1f}%)")
    if text_p  is not None:
        lines.append(f"- Text    : {EMOTIONS[np.argmax(text_p)].upper()} ({text_p.max()*100:.1f}%)"
                     + (f" ['{text_in[:40]}']" if text_in else ""))
    if beh_p   is not None:
        lines.append(f"- Behav   : stress={stress} → {EMOTIONS[np.argmax(beh_p)].upper()}")
    if audio_p is not None:
        lines.append(f"- Audio   : {EMOTIONS[np.argmax(audio_p)].upper()} ({audio_p.max()*100:.1f}%)")

    top3 = sorted(zip(EMOTIONS,fused),key=lambda x:-x[1])[:3]
    mod  = chr(10).join(lines)

    try:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)
        prompt = f"""You are MindSight, an empathetic AI mental health assistant.

ANALYSIS:
- Primary emotion: {emotion.upper()} ({conf*100:.1f}% confidence)
- Emotional state: {state.upper()}
- Top 3: {', '.join(f'{e}({p*100:.1f}%)' for e,p in top3)}
- Modalities: {', '.join(f'{k}={v}' for k,v in (used_w or {}).items())}

SIGNALS:
{mod}

Write a warm mental health report in 4 sections:
1. WHAT WE DETECTED — explain emotion simply
2. YOUR SIGNALS — what each modality revealed
3. MENTAL HEALTH INSIGHT — meaningful wellness insight
4. SUGGESTED ACTIONS — 3 specific things to do right now

Under 300 words. Warm, human, supportive. Speak directly as 'you'."""

        resp = client.chat.completions.create(
            model="llama3-8b-8192",
            messages=[{"role":"user","content":prompt}],
            max_tokens=400)
        return resp.choices[0].message.content
    except Exception as e:
        print(f"Groq error: {e}")
        advice = {"positive":"You seem to be in a good place. Keep nurturing what brings you joy.",
                  "neutral" :"Your state is balanced. A good time to reflect or focus.",
                  "negative":"You may be going through a tough time. Be gentle with yourself."}
        return (f"Detected: {emotion.upper()} ({conf*100:.1f}%)\n"
                f"State: {state.upper()}\n\n{mod}\n\n{advice.get(state,'')}")

print("Report generator ready")


In [ ]:
# CELL 5 — QUESTIONNAIRE
def run_questionnaire():
    print("\n" + "="*50)
    print("  LIFESTYLE QUESTIONNAIRE")
    print("="*50)
    def ask_int(q,lo,hi,d):
        while True:
            try:
                raw=input(f"\n{q} [{lo}-{hi}] (default={d}): ").strip()
                if not raw: return d
                v=int(raw)
                if lo<=v<=hi: return v
            except ValueError: pass
            print(f"  Enter {lo}-{hi}")
    def ask_float(q,lo,hi,d):
        while True:
            try:
                raw=input(f"\n{q} [{lo}-{hi}] (default={d}): ").strip()
                if not raw: return d
                v=float(raw)
                if lo<=v<=hi: return v
            except ValueError: pass
            print(f"  Enter {lo}-{hi}")
    def ask_choice(q,opts,d=0):
        print(f"\n{q}")
        for i,o in enumerate(opts): print(f"  {i+1}. {o}")
        while True:
            try:
                raw=input(f"  Choice (default={d+1}): ").strip()
                if not raw: return d
                v=int(raw)-1
                if 0<=v<len(opts): return v
            except ValueError: pass
    r={}
    r["age"]                     = ask_int("Age?",10,100,22)
    r["gender"]                  = ask_choice("Gender?",["Female","Male","Other"])
    r["occupation"]              = ask_choice("Occupation?",["Student","Employed","Other"])
    r["work_mode"]               = ask_choice("Work mode?",["Remote","In-person","Hybrid","N/A"])
    r["screen_time_hours"]       = ask_float("Total screen time today (hrs)?",0,24,6)
    r["work_screen_hours"]       = ask_float("Work/study screen time (hrs)?",0,24,4)
    r["leisure_screen_hours"]    = ask_float("Leisure screen time (hrs)?",0,24,2)
    r["sleep_hours"]             = ask_float("Sleep last night (hrs)?",0,12,7)
    r["sleep_quality_1_5"]       = ask_int("Sleep quality (1-5)?",1,5,3)
    r["productivity_0_100"]      = ask_int("Productivity today (0-100)?",0,100,60)
    r["exercise_minutes_per_week"]=ask_int("Exercise this week (min)?",0,1000,60)
    r["social_hours_per_week"]   = ask_float("Social time this week (hrs)?",0,100,5)
    print("\nQuestionnaire complete!")
    return r
print("Questionnaire ready")


In [ ]:
# CELL 6 — VISUALISATION
def plot_results(fused,face_p=None,text_p=None,
                 beh_p=None,audio_p=None,
                 emotion=None,state=None,conf=None):
    panels = {}
    if face_p  is not None: panels["Face"]      = face_p
    if text_p  is not None: panels["Text"]      = text_p
    if beh_p   is not None: panels["Behaviour"] = beh_p
    if audio_p is not None: panels["Audio"]     = audio_p
    panels["FUSED ★"]                           = fused

    n   = len(panels)
    fig,axes = plt.subplots(1,n,figsize=(4.5*n,6))
    if n==1: axes=[axes]

    STATE_COL = {"positive":"#4CAF50","neutral":"#9E9E9E","negative":"#F44336"}
    col = STATE_COL.get(state,"#2196F3")
    emoji = {"angry":"😠","disgust":"🤢","fear":"😨",
             "happy":"😊","neutral":"😐","sad":"😢","surprise":"😲"}.get(emotion,"")

    fig.suptitle(f"MindSight  {emoji}  {emotion.upper() if emotion else ''} | "
                 f"{state.upper() if state else ''} ({conf*100:.1f}% confidence)" if conf else "MindSight",
                 fontsize=13,fontweight="bold",color=col,y=1.02)

    for ax,(name,proba) in zip(axes,panels.items()):
        top = int(np.argmax(proba))
        is_fused = "FUSED" in name
        colors = [col if (i==top and is_fused) else
                  "#1565C0" if i==top else
                  "#E0E0E0" if is_fused else "#B3E5FC"
                  for i in range(7)]
        ax.barh(EMOTIONS,proba,color=colors,edgecolor="white",linewidth=0.5)
        ax.set_xlim([0,1.15])
        ax.set_title(f"{name}\n{EMOTIONS[top].upper()} ({proba.max()*100:.1f}%)",
                     fontsize=10,fontweight="bold" if is_fused else "normal",
                     color=col if is_fused else "#333")
        ax.grid(axis="x",alpha=0.2)
        ax.spines[["top","right"]].set_visible(False)
        for i,p in enumerate(proba):
            ax.text(p+0.02,i,f"{p:.3f}",va="center",fontsize=7)

    plt.tight_layout()
    ts2 = datetime.now().strftime("%Y%m%d_%H%M%S")
    plt.savefig(f"/content/mindsight_{ts2}.png",dpi=150,bbox_inches="tight")
    plt.show()
    print(f"Plot saved")
print("Visualisation ready")


In [ ]:
# CELL 7 — MAIN ANALYSIS
from google.colab import files
import IPython.display as ipd
import librosa

def run_mindsight():
    print("\n" + "="*55)
    print("  MINDSIGHT — MENTAL HEALTH ANALYSIS")
    print("  Inputs taken one by one.")
    print("="*55)

    face_p=text_p=beh_p=audio_p=None
    stress=text_in=None

    # Face
    print("\n--- INPUT 1/4: FACE IMAGE ---")
    try:
        if input("Upload face image? (y/n): ").strip().lower()=="y":
            up = files.upload()
            if up:
                fp = list(up.keys())[0]
                img = Image.open(fp).convert("RGB")
                plt.figure(figsize=(3,3)); plt.imshow(img)
                plt.axis("off"); plt.title("Face"); plt.show()
                face_p = predict_face(fp)
                print(f"Face → {EMOTIONS[np.argmax(face_p)].upper()} ({face_p.max()*100:.1f}%)")
    except EOFError: pass

    # Text
    print("\n--- INPUT 2/4: TEXT ---")
    try:
        if input("Describe how you feel? (y/n): ").strip().lower()=="y":
            text_in = input("How do you feel: ").strip()
            if text_in:
                text_p = predict_text(text_in)
                print(f"Text → {EMOTIONS[np.argmax(text_p)].upper()} ({text_p.max()*100:.1f}%)")
    except EOFError: pass

    # Behaviour
    print("\n--- INPUT 3/4: LIFESTYLE ---")
    try:
        if input("Fill lifestyle questionnaire? (y/n): ").strip().lower()=="y":
            beh_p, stress = predict_behaviour(run_questionnaire())
            print(f"Behaviour → stress={stress} → {EMOTIONS[np.argmax(beh_p)].upper()}")
    except EOFError: pass

    # Audio
    print("\n--- INPUT 4/4: AUDIO ---")
    if audio_model is not None:
        try:
            if input("Upload audio file? (y/n): ").strip().lower()=="y":
                up = files.upload()
                if up:
                    af = list(up.keys())[0]
                    w,sr0 = librosa.load(af,sr=None)
                    ipd.display(ipd.Audio(w,rate=sr0))
                    audio_p = predict_audio(af)
                    if audio_p is not None:
                        print(f"Audio → {EMOTIONS[np.argmax(audio_p)].upper()} ({audio_p.max()*100:.1f}%)")
        except EOFError: pass
    else:
        print("Audio model not loaded — skipping")

    if not any(x is not None for x in [face_p,text_p,beh_p,audio_p]):
        print("No inputs provided"); return None

    # Fuse
    fused,used_w = fuse(face_p,text_p,beh_p,audio_p)
    emotion = EMOTIONS[np.argmax(fused)]
    state   = EMOTION_TO_STATE[emotion]
    conf    = float(fused.max())
    emoji   = EMOTION_EMOJI.get(emotion,"")

    print(f"\n{'='*55}")
    print(f"  {emoji} RESULT: {emotion.upper()}")
    print(f"  STATE : {state.upper()}")
    print(f"  CONF  : {conf*100:.1f}%")
    print(f"  WEIGHTS: {used_w}")
    print(f"{'='*55}")

    plot_results(fused,face_p,text_p,beh_p,audio_p,emotion,state,conf)

    print("\nGenerating mental health report...")
    report = generate_report(emotion,state,conf,fused,
                             face_p,text_p,beh_p,audio_p,
                             stress,text_in,used_w)
    print("\n" + "="*55)
    print("  MENTAL HEALTH INSIGHTS")
    print("="*55)
    print(report)

    # Save
    import json as J
    ts2 = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = f"/content/drive/MyDrive/MindSight_Models/test_results/result_{ts2}.json"
    os.makedirs(os.path.dirname(out),exist_ok=True)
    with open(out,"w") as f:
        J.dump({"timestamp":ts2,"emotion":emotion,"state":state,
                "confidence":round(conf,4),"weights":used_w,"report":report,
                "fused":fused.tolist()},f,indent=2)
    print(f"\nSaved: {out}")
    return {"emotion":emotion,"state":state,"confidence":conf,"report":report}

print("Pipeline ready — run Cell 8 to start")


In [ ]:
# CELL 8 — RUN
result = run_mindsight()

In [ ]:
# CELL 9 — CONTINUOUS LOOP
print("Continuous mode — type 'exit' to stop")
while True:
    try:
        cmd = input("\nPress Enter for new analysis | 'exit' to quit: ").strip().lower()
    except EOFError: break
    if cmd=="exit": print("Stopped."); break
    run_mindsight()
